In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/main_data.csv", encoding="utf-8")

/tmp/ipykernel_92720/2256054893.py:1: DtypeWarning: Columns (0,4,5,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/main_data.csv", encoding="utf-8")


In [3]:
import re
from tqdm.auto import tqdm

# Load lexicon
lex = pd.read_csv("crisis_vocabulary.csv")

terms = (
    lex["term"]
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
    .unique()
    .tolist()
)

# Separate single words vs multi-word expressions
single_words = [t for t in terms if " " not in t]
multi_words  = [t for t in terms if " " in t]

print(f"Single words: {len(single_words)} | Multi-word phrases: {len(multi_words)}")

Single words: 70 | Multi-word phrases: 51


In [4]:
# Escape regex characters
single_words_escaped = [re.escape(w) for w in single_words]
multi_words_escaped  = [re.escape(p) for p in multi_words]

# Word-boundary pattern for single words
single_word_pattern = r"\b(" + "|".join(single_words_escaped) + r")\b"

# Phrase pattern (no word boundaries inside phrases)
multi_word_pattern = r"(" + "|".join(multi_words_escaped) + r")"

# Compile regex (case-insensitive)
single_word_regex = re.compile(single_word_pattern, flags=re.IGNORECASE)
multi_word_regex  = re.compile(multi_word_pattern, flags=re.IGNORECASE)


In [5]:
def count_profanity(text):
    if not isinstance(text, str) or not text:
        return 0

    # count single words
    single_hits = single_word_regex.findall(text)

    # count multi-word phrases
    multi_hits = multi_word_regex.findall(text)

    return len(single_hits) + len(multi_hits)


def count_distinct_profanity(text):
    if not isinstance(text, str) or not text:
        return 0

    single_hits = set(m.lower() for m in single_word_regex.findall(text))
    multi_hits  = set(m.lower() for m in multi_word_regex.findall(text))

    return len(single_hits.union(multi_hits))


In [6]:
tqdm.pandas()

df["crisis_count"] = df["text"].progress_apply(count_profanity)
df["crisis_distinct"] = df["text"].progress_apply(count_distinct_profanity)

df[["text", "crisis_count", "crisis_distinct"]].head(10)


  0%|          | 0/71808 [00:00<?, ?it/s]

  0%|          | 0/71808 [00:00<?, ?it/s]

,text,crisis_count,crisis_distinct
0,"When we start talking about the economy, it's ...",2,2
1,It's no secret which groups are hit the hardes...,0,0
2,How can our elderly who have worked so hard to...,0,0
3,I believe that social security is one of this ...,2,2
4,"In contrast, I am committed to an economic pro...",1,1
5,This just doesn't make sense. Older Americans ...,0,0
6,"In fact, if you look around, you'll notice tha...",0,0
7,"More important, in today's unstable economy wi...",1,1
8,"Indeed, I believe that the earnings limitation...",0,0
9,I want to thank you for this opportunity to vi...,0,0


In [7]:
df.sort_values("crisis_count", ascending=False)[
    ["speech_par_id", "name_date", "text", "crisis_count"]
].head(20)


,speech_par_id,name_date,text,crisis_count
22329,2009_11,NaN,Thirty-five years ago Franklin Roosevelt told ...,15
70748,979_998,NaN,"During the war, you remember, when we all knew...",15
39922,924_17,NaN,The American faith has been a faith in the gro...,13
17277,2581_6,NaN,We complain sometimes about interest rates--an...,12
34210,2376_10,NaN,Here are some things we should do to lower inf...,12
16506,1274_9,NaN,That is the kind of program that we should loo...,12
62050,2008-10-14-blue-bell-pennsylvania_3,2008-10-14,I will begin by making certain that the 700 bi...,11
19243,1265_6,NaN,The fact is they are trying to sell you a prog...,10
57392,2004-09-24-education-janesville-wisconsin_82,2004-09-24,"Audience member. We're praying for you, George...",10
19630,1258_6,NaN,The fact is they are trying to sell you a prog...,10


In [8]:
df["crisis_count"].describe()


count    71808.000000
mean         0.371686
std          0.864065
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         15.000000
Name: crisis_count, dtype: float64

In [9]:
df["word_count"] = df["text"].str.split().str.len()
df["crisis_per_100w"] = (
    df["crisis_count"] / df["word_count"] * 100
).fillna(0)


In [10]:
df["crisis_per_100w"].describe()

count    71808.000000
mean         0.423953
std          1.029253
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         17.647059
Name: crisis_per_100w, dtype: float64

In [11]:
df["election_year"] = df["election_date"].astype(str).str[:4].astype("Int64")



In [12]:
set(df["election_year"])

{1952,
 1956,
 1960,
 1964,
 1968,
 1972,
 1976,
 1980,
 1984,
 1988,
 1992,
 1996,
 2000,
 2004,
 2008,
 2012,
 2016,
 2020}

In [13]:
df["person"] = (
    df["person"].astype(str)
    + "_"
    + df["election_year"].astype(str)
)


In [14]:
df_pop = df[df["Populism"] == 1]
df_non = df[df["Populism"] == 0]

n_pop = len(df_pop)
print("Populist rows:", n_pop)

Populist rows: 1861


In [15]:
df_non_sampled = df_non.sample(
    n=n_pop,
    random_state=42  # for reproducibility
)

# Combine
df_balanced = pd.concat([df_pop, df_non_sampled], axis=0).reset_index(drop=True)

print(df_balanced["Populism"].value_counts())


1    1861
0    1861
Name: Populism, dtype: int64


In [16]:
df_balanced[["crisis_per_100w", "Populism"]].groupby("Populism").describe()


crisis_per_100w                                                   
                   count      mean       std  min  25%  50%  75%        max
Populism                                                                   
0                 1861.0  0.413311  1.027229  0.0  0.0  0.0  0.0  11.111111
1                 1861.0  0.299126  0.794803  0.0  0.0  0.0  0.0   9.375000

In [17]:
import statsmodels.api as sm

y = df_balanced["crisis_per_100w"]
X = df_balanced[["Populism"]]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:        crisis_per_100w   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     14.38
Date:                Mon, 15 Dec 2025   Prob (F-statistic):           0.000152
Time:                        17:09:58   Log-Likelihood:                -4963.5
No. Observations:                3722   AIC:                             9931.
Df Residuals:                    3720   BIC:                             9943.
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.4133      0.024     17.353      0.0

In [18]:
import numpy as np
coefs = []

for seed in range(100):
    df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
    df_bal = pd.concat([df_pop, df_non_sampled])

    y = df_bal["crisis_per_100w"]
    X = sm.add_constant(df_bal[["Populism"]])

    res = sm.OLS(y, X).fit(cov_type="HC3")
    coefs.append(res.params["Populism"])

np.mean(coefs), np.std(coefs)


(-0.12414623685164951, 0.0231076207421135)

In [21]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

fe = smf.ols(
    "crisis_per_100w ~ Populism + prior_president + C(person)",
    data=df_balanced
).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

print(fe.summary())


                            OLS Regression Results                            
Dep. Variable:        crisis_per_100w   R-squared:                       0.030
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     2.350
Date:                Mon, 15 Dec 2025   Prob (F-statistic):              0.135
Time:                        17:09:59   Log-Likelihood:                -4914.6
No. Observations:                3722   AIC:                             9899.
Df Residuals:                    3687   BIC:                         1.012e+04
Df Model:                          34                                         
Covariance Type:              cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '


In [24]:
df.to_csv("prof_llm_crisis.csv", encoding="utf-8", index=False)

In [25]:
dfllm = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_crisis_with_similarity.csv", encoding="utf-8")

/tmp/ipykernel_92720/905135519.py:1: DtypeWarning: Columns (0,4,5,21) have mixed types. Specify dtype option on import or set low_memory=False.
  dfllm = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_crisis_with_similarity.csv", encoding="utf-8")


In [27]:
df_pop = dfllm[dfllm["Populism"] == 1]
df_non = dfllm[dfllm["Populism"] == 0]

n_pop = len(df_pop)
print("Populist rows:", n_pop)
df_non_sampled = df_non.sample(
    n=n_pop,
    random_state=42  # for reproducibility
)

# Combine
df_balanced = pd.concat([df_pop, df_non_sampled], axis=0).reset_index(drop=True)

print(df_balanced["Populism"].value_counts())

1    1861
0    1861
Name: Populism, dtype: int64


In [28]:
df_balanced[["agg_sim_top10", "agg_sim_max"]].describe()
df_balanced[["agg_sim_top10", "agg_sim_max"]].corr()


,agg_sim_top10,agg_sim_max
agg_sim_top10,1.000000,0.874727
agg_sim_max,0.874727,1.000000


In [29]:
for var in ["agg_sim_top10", "agg_sim_max"]:
    res = smf.ols(
        f"{var} ~ Populism + prior_president + C(person)",
        data=df_balanced
    ).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})
    print(var, res.params["Populism"], res.pvalues["Populism"])


agg_sim_top10 0.0008397147923138562 0.7640035184496927
agg_sim_max 0.0028521628973474994 0.4998764689940214


In [30]:
import statsmodels.api as sm

y = df_balanced["agg_sim_top10"]
X = sm.add_constant(df_balanced[["Populism"]])

ols = sm.OLS(y, X).fit(cov_type="HC3")
print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:          agg_sim_top10   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     34.92
Date:                Mon, 15 Dec 2025   Prob (F-statistic):           3.75e-09
Time:                        17:14:04   Log-Likelihood:                 5370.9
No. Observations:                3722   AIC:                        -1.074e+04
Df Residuals:                    3720   BIC:                        -1.073e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.2215      0.001    167.849      0.0

In [31]:
import numpy as np

coefs = []

for seed in range(100):
    df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
    df_bal = pd.concat([df_pop, df_non_sampled])

    y = df_bal["agg_sim_top10"]
    X = sm.add_constant(df_bal[["Populism"]])

    res = sm.OLS(y, X).fit()
    coefs.append(res.params["Populism"])

np.mean(coefs), np.std(coefs)

(0.011351471881217046, 0.0013652696636697705)

In [32]:
import statsmodels.api as sm

y = df_balanced["agg_sim_top10"]

X = df_balanced[["Populism", "prior_president"]]
X = sm.add_constant(X)

ols = sm.OLS(y, X).fit(cov_type="HC3")
print(ols.summary())


                            OLS Regression Results                            
Dep. Variable:          agg_sim_top10   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     18.78
Date:                Mon, 15 Dec 2025   Prob (F-statistic):           7.65e-09
Time:                        17:14:16   Log-Likelihood:                 5372.6
No. Observations:                3722   AIC:                        -1.074e+04
Df Residuals:                    3719   BIC:                        -1.072e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.2231      0.002    1

In [33]:
import statsmodels.formula.api as smf

fe_person = smf.ols(
    "agg_sim_top10 ~ Populism + prior_president + C(person)",
    data=df_balanced
).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

print(fe_person.summary())

                            OLS Regression Results                            
Dep. Variable:          agg_sim_top10   R-squared:                       0.093
Model:                            OLS   Adj. R-squared:                  0.085
Method:                 Least Squares   F-statistic:                     8.584
Date:                Mon, 15 Dec 2025   Prob (F-statistic):            0.00611
Time:                        17:14:30   Log-Likelihood:                 5535.8
No. Observations:                3722   AIC:                        -1.100e+04
Df Residuals:                    3687   BIC:                        -1.078e+04
Df Model:                          34                                         
Covariance Type:              cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '


# Real/Constructed Crises


In [34]:
df = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/main_data.csv", encoding="utf-8")

/tmp/ipykernel_92720/2256054893.py:1: DtypeWarning: Columns (0,4,5,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/main_data.csv", encoding="utf-8")


In [36]:
TEXT_COL = "text"

REAL_LEX_PATH = "real_crisis.csv"           # column: term
CONSTR_LEX_PATH = "constructed_crisis.csv"  # column: term


# -----------------------------
# 1) Load lexicons (robust)
# -----------------------------
def load_term_list(path: str) -> list[str]:
    lex_raw = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)

    # If the file got read as a single column (weird header), recover a "term" column
    if lex_raw.shape[1] == 1:
        only_col = lex_raw.columns[0]
        s = lex_raw[only_col].astype(str)

        tmp = (
            s.str.replace("\ufeff", "", regex=False)
             .str.replace('"', "", regex=False)
             .str.strip()
             .str.split(r"[,\t;]", n=1, expand=True)[0]
             .to_frame(name="term")
        )
        tmp["term"] = tmp["term"].astype(str).str.strip()
        tmp = tmp[tmp["term"].str.lower().ne("term") & tmp["term"].ne("")]
        lex = tmp.copy()
    else:
        lex_raw.columns = [c.strip().replace("\ufeff", "") for c in lex_raw.columns]
        if "term" not in lex_raw.columns:
            raise ValueError(f"{path}: expected a 'term' column, found {lex_raw.columns.tolist()}")
        lex = lex_raw.copy()

    terms = (
        lex["term"]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .unique()
        .tolist()
    )
    return terms


real_terms = load_term_list(REAL_LEX_PATH)
constr_terms = load_term_list(CONSTR_LEX_PATH)

print(f"Real-crisis terms: {len(real_terms)} | Constructed-crisis terms: {len(constr_terms)}")


# -----------------------------
# 2) Build regex counters (single vs phrases)
# -----------------------------
def build_regexes(terms: list[str]):
    single_words = [t for t in terms if " " not in t]
    multi_words  = [t for t in terms if " " in t]

    single_words_escaped = [re.escape(w) for w in single_words]
    multi_words_escaped  = [re.escape(p) for p in multi_words]

    single_word_regex = (
        re.compile(r"\b(" + "|".join(single_words_escaped) + r")\b", flags=re.IGNORECASE)
        if single_words_escaped else None
    )
    multi_word_regex = (
        re.compile(r"(" + "|".join(multi_words_escaped) + r")", flags=re.IGNORECASE)
        if multi_words_escaped else None
    )

    return single_word_regex, multi_word_regex, len(single_words), len(multi_words)


real_single_re, real_multi_re, n_real_single, n_real_multi = build_regexes(real_terms)
con_single_re, con_multi_re, n_con_single, n_con_multi = build_regexes(constr_terms)

print(f"Real: single={n_real_single}, phrases={n_real_multi}")
print(f"Constructed: single={n_con_single}, phrases={n_con_multi}")


def count_hits(text, single_re, multi_re):
    if not isinstance(text, str) or not text:
        return 0
    single_hits = single_re.findall(text) if single_re else []
    multi_hits  = multi_re.findall(text) if multi_re else []
    return len(single_hits) + len(multi_hits)


def count_distinct_hits(text, single_re, multi_re):
    if not isinstance(text, str) or not text:
        return 0
    single_hits = set(m.lower() for m in single_re.findall(text)) if single_re else set()
    multi_hits  = set(m.lower() for m in multi_re.findall(text)) if multi_re else set()
    return len(single_hits.union(multi_hits))


# -----------------------------
# 3) Apply to dataframe
# -----------------------------
tqdm.pandas()

df["real_crisis_count"] = df[TEXT_COL].progress_apply(lambda t: count_hits(t, real_single_re, real_multi_re))
df["real_crisis_distinct"] = df[TEXT_COL].progress_apply(lambda t: count_distinct_hits(t, real_single_re, real_multi_re))

df["constructed_crisis_count"] = df[TEXT_COL].progress_apply(lambda t: count_hits(t, con_single_re, con_multi_re))
df["constructed_crisis_distinct"] = df[TEXT_COL].progress_apply(lambda t: count_distinct_hits(t, con_single_re, con_multi_re))

# Normalize per 100 words (like you did before)
df["word_count"] = df[TEXT_COL].fillna("").astype(str).str.split().str.len().clip(lower=1)

df["real_crisis_per_100w"] = (df["real_crisis_count"] / df["word_count"] * 100).fillna(0)
df["constructed_crisis_per_100w"] = (df["constructed_crisis_count"] / df["word_count"] * 100).fillna(0)

# Key rhetorical indicator: crisis inflation / constructed emphasis
df["constructed_minus_real_per_100w"] = df["constructed_crisis_per_100w"] - df["real_crisis_per_100w"]

# Quick check
df[[TEXT_COL,
    "real_crisis_count","constructed_crisis_count",
    "real_crisis_per_100w","constructed_crisis_per_100w",
    "constructed_minus_real_per_100w"]].head(10)

# Top examples
df.sort_values("constructed_crisis_count", ascending=False)[
    ["speech_par_id", "name_date", TEXT_COL, "constructed_crisis_count", "constructed_crisis_per_100w"]
].head(20)

df.sort_values("real_crisis_count", ascending=False)[
    ["speech_par_id", "name_date", TEXT_COL, "real_crisis_count", "real_crisis_per_100w"]
].head(20)

Real-crisis terms: 35 | Constructed-crisis terms: 27
Real: single=14, phrases=21
Constructed: single=4, phrases=23


  0%|          | 0/71808 [00:00<?, ?it/s]

  0%|          | 0/71808 [00:00<?, ?it/s]

  0%|          | 0/71808 [00:00<?, ?it/s]

  0%|          | 0/71808 [00:00<?, ?it/s]

,speech_par_id,name_date,text,real_crisis_count,real_crisis_per_100w
34210,2376_10,NaN,Here are some things we should do to lower inf...,12,4.225352
17277,2581_6,NaN,We complain sometimes about interest rates--an...,11,4.545455
45377,1178_9,NaN,You can answer those questions once and for al...,9,6.976744
29530,2371_13,NaN,The reason unemployment remains high is that t...,9,6.521739
1984,2244_5,NaN,Now the second pledge that the President made ...,9,3.114187
70748,979_998,NaN,"During the war, you remember, when we all knew...",8,0.506329
36176,1871_26,NaN,"I have traveled a great deal, as you know perh...",8,3.404255
42442,3226_48,NaN,Last spring we announced that we would work to...,7,5.185185
10281,2254_5,NaN,Now what is it that keeps a great and decent c...,7,2.430556
5666,2455_12,NaN,Listen to what Prime Minister Jim Callaghan of...,7,3.910615


In [37]:
df["election_year"] = df["election_date"].astype(str).str[:4].astype("Int64")

df["person"] = (
    df["person"].astype(str)
    + "_"
    + df["election_year"].astype(str)
)

df_pop = df[df["Populism"] == 1]
df_non = df[df["Populism"] == 0]

n_pop = len(df_pop)
print("Populist rows:", n_pop)

df_non_sampled = df_non.sample(
    n=n_pop,
    random_state=42  # for reproducibility
)

# Combine
df_balanced = pd.concat([df_pop, df_non_sampled], axis=0).reset_index(drop=True)

print(df_balanced["Populism"].value_counts())


Populist rows: 1861
1    1861
0    1861
Name: Populism, dtype: int64


# Real Crises

In [38]:
import statsmodels.api as sm

y = df_balanced["real_crisis_per_100w"]
X = df_balanced[["Populism"]]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())


                             OLS Regression Results                             
Dep. Variable:     real_crisis_per_100w   R-squared:                       0.007
Model:                              OLS   Adj. R-squared:                  0.006
Method:                   Least Squares   F-statistic:                     24.34
Date:                  Wed, 17 Dec 2025   Prob (F-statistic):           8.41e-07
Time:                          18:18:04   Log-Likelihood:                -2976.6
No. Observations:                  3722   AIC:                             5957.
Df Residuals:                      3720   BIC:                             5970.
Df Model:                             1                                         
Covariance Type:                    HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1867      0.015

In [39]:
import numpy as np
coefs = []

for seed in range(100):
    df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
    df_bal = pd.concat([df_pop, df_non_sampled])

    y = df_bal["real_crisis_per_100w"]
    X = sm.add_constant(df_bal[["Populism"]])

    res = sm.OLS(y, X).fit(cov_type="HC3")
    coefs.append(res.params["Populism"])

np.mean(coefs), np.std(coefs)


(-0.08443307444891061, 0.013156630184129242)

In [40]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

fe = smf.ols(
    "real_crisis_per_100w ~ Populism + prior_president + C(person)",
    data=df_balanced
).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

print(fe.summary())

                             OLS Regression Results                             
Dep. Variable:     real_crisis_per_100w   R-squared:                       0.033
Model:                              OLS   Adj. R-squared:                  0.024
Method:                   Least Squares   F-statistic:                     29.14
Date:                  Wed, 17 Dec 2025   Prob (F-statistic):           5.69e-06
Time:                          18:21:38   Log-Likelihood:                -2925.5
No. Observations:                  3722   AIC:                             5921.
Df Residuals:                      3687   BIC:                             6139.
Df Model:                            34                                         
Covariance Type:                cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '


# Constructed Crises

In [41]:
import statsmodels.api as sm

y = df_balanced["constructed_crisis_count"]
X = df_balanced[["Populism"]]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())

                               OLS Regression Results                               
Dep. Variable:     constructed_crisis_count   R-squared:                       0.000
Model:                                  OLS   Adj. R-squared:                 -0.000
Method:                       Least Squares   F-statistic:                    0.2231
Date:                      Wed, 17 Dec 2025   Prob (F-statistic):              0.637
Time:                              18:21:46   Log-Likelihood:                 4650.0
No. Observations:                      3722   AIC:                            -9296.
Df Residuals:                          3720   BIC:                            -9284.
Df Model:                                 1                                         
Covariance Type:                        HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------

In [42]:
import numpy as np
coefs = []

for seed in range(100):
    df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
    df_bal = pd.concat([df_pop, df_non_sampled])

    y = df_bal["constructed_crisis_count"]
    X = sm.add_constant(df_bal[["Populism"]])

    res = sm.OLS(y, X).fit(cov_type="HC3")
    coefs.append(res.params["Populism"])

np.mean(coefs), np.std(coefs)

(-0.0008597528210640012, 0.0017026307918060143)

In [43]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

fe = smf.ols(
    "constructed_crisis_count ~ Populism + prior_president + C(person)",
    data=df_balanced
).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

print(fe.summary())

                               OLS Regression Results                               
Dep. Variable:     constructed_crisis_count   R-squared:                       0.010
Model:                                  OLS   Adj. R-squared:                  0.001
Method:                       Least Squares   F-statistic:                   0.01219
Date:                      Wed, 17 Dec 2025   Prob (F-statistic):              0.913
Time:                              18:21:53   Log-Likelihood:                 4668.3
No. Observations:                      3722   AIC:                            -9267.
Df Residuals:                          3687   BIC:                            -9049.
Df Model:                                34                                         
Covariance Type:                    cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '


# LLM

In [44]:
dfllm = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_crisis_with_similarity_real_constructed.csv", encoding="utf-8")

/tmp/ipykernel_92720/2267522352.py:1: DtypeWarning: Columns (0,4,5,21) have mixed types. Specify dtype option on import or set low_memory=False.
  dfllm = pd.read_csv("/home/scc/sergio.zanotto/bin/badmanners/vulgarity/prof_llm_crisis_with_similarity_real_constructed.csv", encoding="utf-8")


In [45]:
df_pop = dfllm[dfllm["Populism"] == 1]
df_non = dfllm[dfllm["Populism"] == 0]

n_pop = len(df_pop)
print("Populist rows:", n_pop)
df_non_sampled = df_non.sample(
    n=n_pop,
    random_state=42  # for reproducibility
)

# Combine
df_balanced = pd.concat([df_pop, df_non_sampled], axis=0).reset_index(drop=True)

print(df_balanced["Populism"].value_counts())

Populist rows: 1861
1    1861
0    1861
Name: Populism, dtype: int64


# Real

In [46]:
import statsmodels.api as sm

y = df_balanced["real_sim_top10"]
X = sm.add_constant(df_balanced[["Populism"]])

ols = sm.OLS(y, X).fit(cov_type="HC3")
print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:         real_sim_top10   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     36.89
Date:                Wed, 17 Dec 2025   Prob (F-statistic):           1.38e-09
Time:                        18:34:24   Log-Likelihood:                 4738.1
No. Observations:                3722   AIC:                            -9472.
Df Residuals:                    3720   BIC:                            -9460.
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1816      0.002    119.234      0.0

In [47]:
import numpy as np

coefs = []

for seed in range(100):
    df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
    df_bal = pd.concat([df_pop, df_non_sampled])

    y = df_bal["real_sim_top10"]
    X = sm.add_constant(df_bal[["Populism"]])

    res = sm.OLS(y, X).fit()
    coefs.append(res.params["Populism"])

np.mean(coefs), np.std(coefs)

(0.013904527289780729, 0.0015195562350492138)

In [48]:
import statsmodels.api as sm

y = df_balanced["real_sim_top10"]

X = df_balanced[["Populism", "prior_president"]]
X = sm.add_constant(X)

ols = sm.OLS(y, X).fit(cov_type="HC3")
print(ols.summary())


                            OLS Regression Results                            
Dep. Variable:         real_sim_top10   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     18.69
Date:                Wed, 17 Dec 2025   Prob (F-statistic):           8.35e-09
Time:                        18:34:34   Log-Likelihood:                 4738.5
No. Observations:                3722   AIC:                            -9471.
Df Residuals:                    3719   BIC:                            -9452.
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.1826      0.002     

In [49]:
import statsmodels.formula.api as smf

fe_person = smf.ols(
    "real_sim_top10 ~ Populism + prior_president + C(person)",
    data=df_balanced
).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

print(fe_person.summary())

                            OLS Regression Results                            
Dep. Variable:         real_sim_top10   R-squared:                       0.123
Model:                            OLS   Adj. R-squared:                  0.115
Method:                 Least Squares   F-statistic:                     25.62
Date:                Wed, 17 Dec 2025   Prob (F-statistic):           1.54e-05
Time:                        18:34:37   Log-Likelihood:                 4963.1
No. Observations:                3722   AIC:                            -9856.
Df Residuals:                    3687   BIC:                            -9638.
Df Model:                          34                                         
Covariance Type:              cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '


# Contructed

In [50]:
import statsmodels.api as sm

y = df_balanced["constructed_sim_top10"]
X = sm.add_constant(df_balanced[["Populism"]])

ols = sm.OLS(y, X).fit(cov_type="HC3")
print(ols.summary())

                              OLS Regression Results                             
Dep. Variable:     constructed_sim_top10   R-squared:                       0.001
Model:                               OLS   Adj. R-squared:                  0.001
Method:                    Least Squares   F-statistic:                     3.630
Date:                   Wed, 17 Dec 2025   Prob (F-statistic):             0.0568
Time:                           18:34:48   Log-Likelihood:                 5522.1
No. Observations:                   3722   AIC:                        -1.104e+04
Df Residuals:                       3720   BIC:                        -1.103e+04
Df Model:                              1                                         
Covariance Type:                     HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1812 

In [51]:
import numpy as np

coefs = []

for seed in range(100):
    df_non_sampled = df_non.sample(n=n_pop, random_state=seed)
    df_bal = pd.concat([df_pop, df_non_sampled])

    y = df_bal["constructed_sim_top10"]
    X = sm.add_constant(df_bal[["Populism"]])

    res = sm.OLS(y, X).fit()
    coefs.append(res.params["Populism"])

np.mean(coefs), np.std(coefs)

(0.003279294363427258, 0.0012257037183484871)

In [52]:
import statsmodels.api as sm

y = df_balanced["constructed_sim_top10"]

X = df_balanced[["Populism", "prior_president"]]
X = sm.add_constant(X)

ols = sm.OLS(y, X).fit(cov_type="HC3")
print(ols.summary())


                              OLS Regression Results                             
Dep. Variable:     constructed_sim_top10   R-squared:                       0.004
Model:                               OLS   Adj. R-squared:                  0.003
Method:                    Least Squares   F-statistic:                     6.713
Date:                   Wed, 17 Dec 2025   Prob (F-statistic):            0.00123
Time:                           18:34:56   Log-Likelihood:                 5527.2
No. Observations:                   3722   AIC:                        -1.105e+04
Df Residuals:                       3719   BIC:                        -1.103e+04
Df Model:                              2                                         
Covariance Type:                     HC3                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const       

In [53]:
import statsmodels.formula.api as smf

fe_person = smf.ols(
    "constructed_sim_top10 ~ Populism + prior_president + C(person)",
    data=df_balanced
).fit(cov_type="cluster", cov_kwds={"groups": df_balanced["person"]})

print(fe_person.summary())

                              OLS Regression Results                             
Dep. Variable:     constructed_sim_top10   R-squared:                       0.058
Model:                               OLS   Adj. R-squared:                  0.050
Method:                    Least Squares   F-statistic:                     4.570
Date:                   Wed, 17 Dec 2025   Prob (F-statistic):             0.0400
Time:                           18:35:01   Log-Likelihood:                 5632.4
No. Observations:                   3722   AIC:                        -1.119e+04
Df Residuals:                       3687   BIC:                        -1.098e+04
Df Model:                             34                                         
Covariance Type:                 cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

/usr/local/jupyterhub/lib64/python3.10/site-packages/statsmodels/base/model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 1
  warnings.warn('covariance of constraints does not have full '
